In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import os
from dotenv import load_dotenv
from pprint import pprint

load_dotenv()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

# Groq (gpt-oss), paced to respect the free-tier rate limit
from math_assistant_agent.enrichment import (
    build_groq_client, extract_graph_entities_groq, enrich_graph_records
)


In [4]:
client = build_groq_client(api_key=GROQ_API_KEY)

In [7]:
graph_data_enriched = enrich_graph_records(
    "/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_2026-07-17-16.json", 
    client=client, extract_fn=extract_graph_entities_groq, sleep_seconds=15
)

Extracted: Complex Analysis | ['Residue theorem', 'Argument principle', 'Polynomial factorization', 'Uniform distribution on unit circle', 'Random polynomials']
Error extracting data with Groq: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '{"domain":"Number Theory","concepts":["Anabelian Geometry","Neukirch–Uchida Theorem","Etale Fundamental Group","Section Conjecture","ABC Conjecture"],"resolution_steps":[{"step_number":1,"description":"Mochizuki’s work is framed within Grothendieck’s anabelian program, which aims to recover arithmetic information from profinite fundamental groups; this perspective is hoped to shed light on the ABC conjecture.","formula_latex":""},{"step_number":2,"description":"The Neukirch–Uchida theorem states that an isomorphism of absolute Galois groups of number fields extends to an inne

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kxsa2evmf07948nc17h47ss2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197446, Requested 3760. Please try again in 8m40.991999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [11]:
import gc

candidates = [
    obj for obj in gc.get_objects()
    if isinstance(obj, dict) and set(obj.keys()) == {"nodes", "edges"}
    and isinstance(obj.get("nodes"), list)
]

for i, c in enumerate(candidates):
    print(i, len(c["nodes"]), "nodes,", len(c["edges"]), "edges")

0 1225 nodes, 1670 edges
1 357 nodes, 452 edges


In [12]:
def enriched_question_count(graph_data):
    question_ids = {n["id"] for n in graph_data["nodes"] if n["label"] == "Question"}
    enriched = {e["source"] for e in graph_data["edges"] if e["type"] == "HAS_FIRST_STEP"}
    return len(question_ids & enriched), len(question_ids)

def label_counts(graph_data):
    counts = {}
    for n in graph_data["nodes"]:
        counts[n["label"]] = counts.get(n["label"], 0) + 1
    return counts

def has_duplicate_ids(graph_data):
    ids = [n["id"] for n in graph_data["nodes"]]
    return len(ids) != len(set(ids))

for i, c in enumerate(candidates):
    enriched, total = enriched_question_count(c)
    print(f"candidate {i}: {enriched}/{total} questions enriched, "
        f"labels={label_counts(c)}, duplicate_ids={has_duplicate_ids(c)}")

candidate 0: 80/100 questions enriched, labels={'Question': 100, 'Answer': 100, 'Tag': 139, 'Domain': 46, 'Concept': 384, 'ResolutionStep': 456}, duplicate_ids=False
candidate 1: 1/100 questions enriched, labels={'Question': 100, 'Answer': 100, 'Tag': 139, 'Domain': 1, 'Concept': 5, 'ResolutionStep': 12}, duplicate_ids=False


In [ ]:
from math_assistant_agent.data import save_graph_json




graph_data_enriched = enrich_graph_records(
    "/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_2026-07-17-16.json",
    client=client, extract_fn=extract_graph_entities_groq, sleep_seconds=15
)

In [13]:
from math_assistant_agent.data import save_graph_json
from datetime import datetime

GRAPH_PATH = "/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data"
INGESTION_DATE = datetime.now().strftime("%Y-%m-%d-%H")

best = candidates[0]
save_graph_json(best, path=f"{GRAPH_PATH}/graph_math_kb_{INGESTION_DATE}.json")

# save_graph_json(graph_data_enriched, path=f"{GRAPH_PATH}/graph_math_kb_{INGESTION_DATE}.json")

✅ Grafo de conhecimento salvo em: /home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-17-23.json


'/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-17-23.json'

In [16]:
from math_assistant_agent.visualization import render_graph

render_graph(best, output_path=f"{GRAPH_PATH}/graph_math_kb_{INGESTION_DATE}.html")

Rendering visualization for 1225 nodes and 1670 edges...
✅ Graph rendered successfully! Open '/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-17-23.html' in your browser.
